In [1]:
import pandas as pd

In [2]:
# Test queries for RAG evaluation
# Organized by difficulty and emotional coverage

TEST_QUERIES = [
    # ═══════════════════════════════════════════════════════
    # EASY: Direct keyword/topic matches
    # ═══════════════════════════════════════════════════════
    {
        "id": "Q001",
        "query": "I need help setting boundaries with my family",
        "expected_episodes": ["001"],  # Mark Manson boundaries
        "expected_themes": ["boundaries", "saying no", "relationships"],
        "difficulty": "easy",
        "emotion": "frustrated",
    },
    {
        "id": "Q002",
        "query": "I keep comparing myself to people on social media and feeling inadequate",
        "expected_episodes": ["017"],  # Comparing yourself to others
        "expected_themes": ["comparison", "social media", "self-worth"],
        "difficulty": "easy",
        "emotion": "inadequate",
    },
    {
        "id": "Q003",
        "query": "I'm scared of being judged and it's blocking my creativity",
        "expected_episodes": ["010"],  # Fear of being judged
        "expected_themes": ["judgment", "creativity", "fear"],
        "difficulty": "easy",
        "emotion": "fear",
    },
    
    # ═══════════════════════════════════════════════════════
    # MEDIUM: Semantic understanding required
    # ═══════════════════════════════════════════════════════
    {
        "id": "Q004",
        "query": "I keep beating myself up over mistakes I made at work",
        "expected_episodes": ["002", "003"],  # Self-compassion + Trusting yourself
        "expected_themes": ["self-compassion", "shame", "mistakes", "self-criticism"],
        "difficulty": "medium",
        "emotion": "shame",
    },
    {
        "id": "Q005",
        "query": "Why do I sabotage good relationships when things are going well?",
        "expected_episodes": ["004", "033"],  # Self-sabotage + Self-abandonment
        "expected_themes": ["self-sabotage", "relationships", "patterns"],
        "difficulty": "medium",
        "emotion": "confused",
    },
    {
        "id": "Q006",
        "query": "I feel stuck in life and don't know what direction to take",
        "expected_episodes": ["011", "026"],  # Life feels empty + feeling lost
        "expected_themes": ["lost", "direction", "purpose", "uncertainty"],
        "difficulty": "medium",
        "emotion": "lost",
    },
    {
        "id": "Q007",
        "query": "I'm constantly anxious and my mind won't stop racing",
        "expected_episodes": ["019", "020", "036", "039"],  # Stopping anxiety (multiple)
        "expected_themes": ["anxiety", "overthinking", "calm", "nervous system"],
        "difficulty": "medium",
        "emotion": "anxious",
    },
    {
        "id": "Q008",
        "query": "I feel lonely even though I'm surrounded by people",
        "expected_episodes": ["018"],  # Simon Sinek on loneliness
        "expected_themes": ["loneliness", "connection", "isolation"],
        "difficulty": "medium",
        "emotion": "lonely",
    },
    
    # ═══════════════════════════════════════════════════════
    # HARD: Requires inference and emotional understanding
    # ═══════════════════════════════════════════════════════
    {
        "id": "Q009",
        "query": "People always take advantage of me and I can't say no",
        "expected_episodes": ["001", "033"],  # Boundaries + people-pleasing
        "expected_themes": ["boundaries", "people-pleasing", "self-worth"],
        "difficulty": "hard",
        "emotion": "resentful",
    },
    {
        "id": "Q010",
        "query": "I don't feel good enough no matter what I achieve",
        "expected_episodes": ["008", "034", "035"],  # Perfectionism + self-love
        "expected_themes": ["perfectionism", "self-worth", "achievement", "enough"],
        "difficulty": "hard",
        "emotion": "inadequate",
    },
    {
        "id": "Q011",
        "query": "I'm grieving a breakup and can't stop thinking about what I did wrong",
        "expected_episodes": ["025", "028"],  # Grief + marriage/relationships
        "expected_themes": ["grief", "heartbreak", "relationships", "self-blame"],
        "difficulty": "hard",
        "emotion": "heartbroken",
    },
    {
        "id": "Q012",
        "query": "I want to make a change but I'm terrified of failing",
        "expected_episodes": ["009", "016", "023"],  # Transform life + fear + overcome fear
        "expected_themes": ["change", "fear", "failure", "courage"],
        "difficulty": "hard",
        "emotion": "afraid",
    },
    
    # ═══════════════════════════════════════════════════════
    # EDGE CASES: Test system limits
    # ═══════════════════════════════════════════════════════
    {
        "id": "Q013",
        "query": "I'm tired all the time and feel unmotivated to do anything",
        "expected_episodes": ["026", "027", "030"],  # Lost/lazy/unmotivated + trauma
        "expected_themes": ["burnout", "fatigue", "motivation", "trauma"],
        "difficulty": "hard",
        "emotion": "exhausted",
        "note": "Tests if system connects fatigue to deeper emotional/trauma issues"
    },
    {
        "id": "Q014",
        "query": "I need to build better habits but keep falling back into old patterns",
        "expected_episodes": ["007", "029"],  # Negative patterns + habits
        "expected_themes": ["habits", "patterns", "change", "consistency"],
        "difficulty": "medium",
        "emotion": "frustrated",
    },
    {
        "id": "Q015",
        "query": "I want to love myself but I don't know where to start",
        "expected_episodes": ["034", "035"],  # How to love yourself (2 episodes)
        "expected_themes": ["self-love", "self-worth", "self-acceptance"],
        "difficulty": "easy",
        "emotion": "confused",
        "note": "Tests if duplicate episodes are handled (ep 034 and 035 both on self-love)"
    },
]

In [14]:
# notebooks/evaluation.ipynb

import sys
import os

sys.path.append(os.path.abspath(".."))

from scripts.rag_pipeline import run_pipeline
from src.vector_store import get_collection
from src.memory import ConversationMemory
from src.data_loader import load_episodes
from src.hybrid_search import HybridSearcher

collection = get_collection()
df = load_episodes()
memory = ConversationMemory()

results = []

for test_case in TEST_QUERIES:
    print(f"\n{'='*60}")
    print(f"Query {test_case['id']}: {test_case['query']}")
    print(f"Expected: {test_case['expected_episodes']}")
    print(f"Difficulty: {test_case['difficulty']}")
    
    # Run pipeline
    output = run_pipeline(
        user_query=test_case['query'],
        collection=collection,
        memory=memory,
        df=df,
        top_k=5,
        search_method="hybrid",
        semantic_weight=0.6,  # 70% semantic, 30% BM25
    )
    
    # Extract top episode IDs
    retrieved = [r['metadata']['episode_id'] for r in output['recommendations'][:5]]
    
    # Calculate Precision@3
    relevant_in_top3 = len(set(retrieved[:3]) & set(test_case['expected_episodes']))
    precision_3 = relevant_in_top3 / 3
    
    # Calculate Recall@5
    recall_5 = len(set(retrieved) & set(test_case['expected_episodes'])) / len(test_case['expected_episodes'])
    
    # MRR
    mrr = 0
    for i, ep_id in enumerate(retrieved[:5], 1):
        if ep_id in test_case['expected_episodes']:
            mrr = 1 / i
            break

    print(f"  Expected: {test_case['expected_episodes']}")
    print(f"  Retrieved: {retrieved[:3]}")
    print(f"  P@3: {precision_3:.2f} | R@5: {recall_5:.2f} | MRR: {mrr:.2f}")
    
    results.append({
        'query_id': test_case['id'],
        'query': test_case['query'],
        'difficulty': test_case['difficulty'],
        'emotion': test_case['emotion'],
        'method': 'hybrid_70_30',
        'semantic_weight': 0.7,
        'precision_3': precision_3,
        'recall_5': recall_5,
        'mrr': mrr,
        'retrieved_top3': str(retrieved[:3]),
        'expected': str(test_case['expected_episodes']),
    })
    memory.clear()

# Summary
df_results = pd.DataFrame(results)
print("\n" + "="*60)
print("OVERALL RESULTS")
print("="*60)
print(f"Average Precision@3: {df_results['precision_3'].mean():.3f}")
print(f"Average Recall@5: {df_results['recall_5'].mean():.3f}")
print(f"\nBy Difficulty:")
print(df_results.groupby('difficulty')['precision_3'].mean())
print(f"Average MRR:         {df_results['mrr'].mean():.3f}")

✓ Loaded 'podcast_episodes' (44 episodes)
✓ embedding type: <class 'list'> <class 'float'>
✓ Loaded 46 episodes

Query Q001: I need help setting boundaries with my family
Expected: ['001']
Difficulty: easy


INFO     | building BM25 index..
INFO     | ✓ BM25 index built for 46 documents
INFO     | Hybrid search returned 5 results (top score: 0.642)


  Expected: ['001']
  Retrieved: ['001', '026', '032']
  P@3: 0.33 | R@5: 1.00 | MRR: 1.00

Query Q002: I keep comparing myself to people on social media and feeling inadequate
Expected: ['017']
Difficulty: easy


INFO     | building BM25 index..
INFO     | ✓ BM25 index built for 46 documents
INFO     | Hybrid search returned 5 results (top score: 0.696)


  Expected: ['017']
  Retrieved: ['017', '032', '018']
  P@3: 0.33 | R@5: 1.00 | MRR: 1.00

Query Q003: I'm scared of being judged and it's blocking my creativity
Expected: ['010']
Difficulty: easy


...

  Expected: ['026', '027', '030']
  Retrieved: ['015', '036', '006']
  P@3: 0.00 | R@5: 0.00 | MRR: 0.00

Query Q014: I need to build better habits but keep falling back into old patterns
Expected: ['007', '029']
Difficulty: medium


INFO     | building BM25 index..
INFO     | ✓ BM25 index built for 46 documents
INFO     | Hybrid search returned 5 results (top score: 0.590)


  Expected: ['007', '029']
  Retrieved: ['028', '032', '043']
  P@3: 0.00 | R@5: 0.00 | MRR: 0.00

Query Q015: I want to love myself but I don't know where to start
Expected: ['034', '035']
Difficulty: easy


INFO     | building BM25 index..
INFO     | ✓ BM25 index built for 46 documents
INFO     | Hybrid search returned 5 results (top score: 0.646)


  Expected: ['034', '035']
  Retrieved: ['034', '002', '033']
  P@3: 0.33 | R@5: 0.50 | MRR: 1.00

OVERALL RESULTS
Average Precision@3: 0.244
Average Recall@5: 0.478

By Difficulty:
difficulty
easy      0.333333
hard      0.200000
medium    0.222222
Name: precision_3, dtype: float64
Average MRR:         0.633


create experiment object to log the evaluation metrics 

In [15]:
import json
from datetime import datetime

experiment = {
    "experiment_id": f"exp_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    "experiment_name":"hybrid search with semantic weight =0.6",
    "retriever": "embedding_only",
    "embedding_model": "text-embedding-3-small",
    "interpret_prompt_version": "v1",
    "explanation_prompt_version": "v1",
    "dataset_size": collection.count(),
    "queries": len(TEST_QUERIES),
    "precision@3": round(df_results["precision_3"].mean(), 3),
    "recall@5": round(df_results["recall_5"].mean(), 3),
    "timestamp": datetime.now().isoformat()
}

logging in json

In [16]:
log_path = "../data/evaluation/retrieval_accuracy.json"

# Load existing experiments
try:
    with open(log_path, "r") as f:
        experiments = json.load(f)
except:
    experiments = []

# Append new experiment
experiments.append(experiment)

# Save updated log
with open(log_path, "w") as f:
    json.dump(experiments, f, indent=2)

print("Experiment logged successfully")

Experiment logged successfully


In [18]:
# notebooks/diagnostic_analysis.ipynb

import pandas as pd
from scripts.rag_pipeline import run_pipeline
from src.vector_store import get_collection
from src.data_loader import load_episodes
from src.memory import ConversationMemory


collection = get_collection()
df = load_episodes()
memory = ConversationMemory()

print("="*60)
print("DIAGNOSTIC: What's Being Retrieved?")
print("="*60)

for test_case in TEST_QUERIES[:5]:  # Analyze first 5 queries
    print(f"\n{'─'*60}")
    print(f"Query: {test_case['query']}")
    print(f"Expected: {test_case['expected_episodes']}")
    
    output = run_pipeline(
        user_query=test_case['query'],
        collection=collection,
        memory=memory,
        df=df,
        search_method="hybrid",
        semantic_weight=0.8,
    )
    
    print(f"\nTop 5 Retrieved:")
    for i, rec in enumerate(output['recommendations'][:5], 1):
        ep_id = rec['metadata']['episode_id']
        similarity = rec['similarity']
        title = rec['metadata']['episode_title'][:50]
        
        is_relevant = "✓" if ep_id in test_case['expected_episodes'] else "✗"
        
        print(f"  {i}. [{is_relevant}] {ep_id} (score: {similarity:.3f})")
        print(f"      {title}...")
    
    # Check if ANY relevant episode appears in top 10
    top_10 = [r['metadata']['episode_id'] for r in output['recommendations'][:10]]
    relevant_in_10 = set(top_10) & set(test_case['expected_episodes'])
    
    if not relevant_in_10:
        print(f"\n  ⚠️  WARNING: No relevant episodes in top 10!")
        print(f"  This suggests retrieval is fundamentally broken for this query.")
    
    memory.clear()

✓ Loaded 'podcast_episodes' (44 episodes)


✓ embedding type: <class 'list'> <class 'float'>
✓ Loaded 46 episodes
DIAGNOSTIC: What's Being Retrieved?

────────────────────────────────────────────────────────────
Query: I need help setting boundaries with my family
Expected: ['001']


INFO     | building BM25 index..
INFO     | ✓ BM25 index built for 46 documents
INFO     | Hybrid search returned 5 results (top score: 0.522)



Top 5 Retrieved:
  1. [✓] 001 (score: 0.522)
      How to Set (And Keep) Your Boundaries This Year, S...
  2. [✗] 032 (score: 0.413)
      #1 CONFIDENCE Expert: The Brutally EASY Way to Eli...
  3. [✗] 026 (score: 0.409)
      The Shocking Reason You're Tired, Lost & Doubting ...
  4. [✗] 027 (score: 0.388)
      This Is Why You FEEL LOST, LAZY & UNMOTIVATED In L...
  5. [✗] 037 (score: 0.375)
      How to Understand Emotions | Dr. Lisa Feldman Barr...

────────────────────────────────────────────────────────────
Query: I keep comparing myself to people on social media and feeling inadequate
Expected: ['017']


INFO     | building BM25 index..
INFO     | ✓ BM25 index built for 46 documents
INFO     | Hybrid search returned 5 results (top score: 0.595)


...


Top 5 Retrieved:
  1. [✓] 010 (score: 0.543)
      Blocked by Fear of Being Judged? Here's How to STO...
  2. [✗] 037 (score: 0.429)
      How to Understand Emotions | Dr. Lisa Feldman Barr...
  3. [✗] 035 (score: 0.428)
      If You Want To LOVE YOURSELF To The Core, WATCH TH...
  4. [✗] 002 (score: 0.425)
      Self-Compassion: How to Make it Work for You | Dr....
  5. [✗] 007 (score: 0.422)
      How To Fix Your Negative Patterns - Alain de Botto...

────────────────────────────────────────────────────────────
Query: I keep beating myself up over mistakes I made at work
Expected: ['002', '003']


INFO     | building BM25 index..
INFO     | ✓ BM25 index built for 46 documents
INFO     | Hybrid search returned 5 results (top score: 0.571)



Top 5 Retrieved:
  1. [✓] 002 (score: 0.571)
      Self-Compassion: How to Make it Work for You | Dr....
  2. [✗] 008 (score: 0.465)
      Understanding the Psychology of Perfectionism - Dr...
  3. [✗] 019 (score: 0.456)
      The Secret to Stopping Anxiety & Fear (That Actual...
  4. [✗] 038 (score: 0.450)
      The ROOT CAUSE Of Trauma & Why You FEEL LOST In Li...
  5. [✓] 003 (score: 0.441)
      Trusting Yourself: How to Stop Self-Abandonment | ...

────────────────────────────────────────────────────────────
Query: Why do I sabotage good relationships when things are going well?
Expected: ['004', '033']


INFO     | building BM25 index..
INFO     | ✓ BM25 index built for 46 documents
INFO     | Hybrid search returned 5 results (top score: 0.492)



Top 5 Retrieved:
  1. [✓] 004 (score: 0.492)
      Stop Self-Sabotage: How to Get Out of Your Own Way...
  2. [✗] 032 (score: 0.450)
      #1 CONFIDENCE Expert: The Brutally EASY Way to Eli...
  3. [✗] 007 (score: 0.442)
      How To Fix Your Negative Patterns - Alain de Botto...
  4. [✗] 001 (score: 0.430)
      How to Set (And Keep) Your Boundaries This Year, S...
  5. [✗] 018 (score: 0.399)
      Simon Sinek: "I FEEL LONELY!" How To Deal With Lon...
